# The validation framework

On real data there are no ground-truth annotator groups, so we cannot check directly whether a method finds the "right" ones. The validation framework builds **synthetic annotations with known groups** on top of [HateCheck](https://huggingface.co/datasets/Paul/hatecheck): each group follows simple rules (e.g., "label every item targeting women as non-hateful") and otherwise uses the gold labels.

This notebook shows how to build benchmark instances, access their ground truth, and evaluate any grouping method with the three analyses of the paper: grouping, annotation prediction and most relevant items. It requires HateCheck: `python data/download_data.py`.

In [ ]:
import os
import sys
import pathlib
import warnings

os.environ['TQDM_DISABLE'] = '1'  # hide progress bars
warnings.filterwarnings('ignore', message='IProgress not found')
warnings.filterwarnings('ignore', category=SyntaxWarning)

ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score

from exp_config import RULES, VALIDATION_VARIANTS, build_group_templates, build_validation_experiments
from exp_utils import HateCheckData
from flocke_utils import build_ground_truth_df
from metrics import clustering_performance

HATECHECK_CSV = ROOT / 'data' / 'hatecheck' / 'test.csv'

## 1. Groups and experiments

A group template is a list of rules from `src/exp_config.py`, and the template `null` follows the gold labels. `build_group_templates()` enumerates all combinations of the validation rules, and `build_validation_experiments(variant)` packs them into experiments with 2 to 5 groups. The three variants share the same groups:
- `balanced`: groups of equal size;
- `imbalanced`: random group sizes;
- `bias`: in addition, each annotator flips some labels on one random HateCheck functionality.

In [ ]:
group_templates = build_group_templates()
experiments = {variant: build_validation_experiments(variant, group_templates=group_templates) for variant in VALIDATION_VARIANTS}
print({variant: len(exps) for variant, exps in experiments.items()}, 'experiments')

VERSION = list(experiments['balanced'])[1]
for group in experiments['balanced'][VERSION]['groups']:
    rules = [f"{rule} -> {RULES[rule]['label']}" for rule in group_templates[group]]
    print(f'group {group}:', rules or 'gold labels')

{'balanced': 163, 'imbalanced': 163, 'bias': 163} experiments
group null: gold labels
group 10: ['immigrant_non_hate -> non-hateful', 'gay_non_hate -> non-hateful']
group 91: ['women_non_hate -> non-hateful', 'gay_non_hate -> non-hateful', 'muslim_non_hate -> non-hateful', 'black_non_hate -> non-hateful']


## 2. A benchmark instance

`HateCheckData` creates `num_annotators` synthetic annotators split among the groups of an experiment, and gives each HateCheck item `annotations_per_entry` labels from randomly drawn annotators. Items have the same format used by FLOCKE: `{item_id: {'text', 'annotations', 'soft_label', ...}}`.

To also hold out items for evaluation, pass `splits=['train', 'test']`, `split_criterion='by_item'` and `split_ratios={'train': 0.8, 'test': 0.2}`.

In [ ]:
def make_dataset(variant, version, num_annotators=20, annotations_per_entry=5):
    parameters = {'version': version, 'num_annotators': num_annotators,
                  'annotations_per_entry': annotations_per_entry, 'random_noise_probability': 0.0}
    return HateCheckData(hatecheck_csv_path=str(HATECHECK_CSV), parameters=parameters,
                         experiments=experiments[variant], group_templates=group_templates)

dataset = make_dataset('balanced', VERSION)
train_items = dataset.get_split('train')
print(len(train_items), 'items')
next(iter(train_items.values()))

3728 items


{'id': 1,
 'annotation task': 'hate speech detection',
 'text': 'I hate women. ',
 'number of annotators': 5,
 'annotators': 'Ann1,Ann17,Ann4,Ann8,Ann9',
 'number of annotations': 5,
 'annotations': {'Ann4': '1',
  'Ann1': '1',
  'Ann9': '1',
  'Ann8': '1',
  'Ann17': '0'},
 'soft_label': {'0': 0.2, '1': 0.8}}

**Ground truth.** Three pieces of ground truth are available, one for each analysis:
- `dataset.groups`: the annotators of each group;
- `build_ground_truth_df(dataset)`: the label each group gives to each item;
- `dataset._disagreement_df`: the items on which each pair of groups disagrees (the *most relevant items*).

In [ ]:
print({group: len(annotators) for group, annotators in dataset.groups.items()})
build_ground_truth_df(dataset).head()

{'null': 7, '10': 7, '91': 6}


,example_id,null_label,10_label,91_label
0,1,1,1,0
1,2,1,1,1
2,3,1,0,0
3,4,1,1,0
4,5,1,1,1


In [ ]:
dataset._disagreement_df.head()

,id,text,disagreeing_group_pairs,null_label,10_label,91_label
0,1,I hate women.,"[(null, 91), (10, 91)]",hateful,hateful,non-hateful
1,3,I hate gay people.,"[(null, 10), (null, 91)]",hateful,non-hateful,non-hateful
2,4,I hate black people.,"[(null, 91), (10, 91)]",hateful,hateful,non-hateful
3,6,I hate Muslims.,"[(null, 91), (10, 91)]",hateful,hateful,non-hateful
4,7,I hate immigrants.,"[(null, 10), (10, 91)]",hateful,non-hateful,hateful


## 3. Plugging in a method

A method receives the training items and the number of groups, and returns a dictionary with:
- `groups`: `{group_id: [annotators]}`;
- `group_labels`: a DataFrame with `example_id` and one column `group{g}` per group, with the 0/1 label that group would give to each item;
- `pair_scores`: a DataFrame with `example_id` and one column `g{a}_g{b}` per pair of groups, scoring how much the two groups disagree on each item.

`evaluate` scores these outputs against the ground truth. Predicted and true groups are matched with the Hungarian algorithm. It is equivalent to the functions that `validation.py` uses for FLOCKE (`hatecheck_grouping`, `group_annotation_f1`, `most_relevant_items_f1` in `src/flocke_utils.py`), which take a trained FLOCKE model instead.

In [ ]:
def evaluate(dataset, output) -> dict:
    # Grouping: accuracy (after optimal matching), purity and completeness, on the annotators the method has seen
    observed = {a for annotators in output['groups'].values() for a in annotators}
    gt_groups = {name: [a for a in annotators if a in observed] for name, annotators in dataset.groups.items()}
    grouping = clustering_performance(gt_groups, output['groups'])

    # Annotation prediction: macro-F1 between each predicted group's labels and those of the matched true group
    merged = build_ground_truth_df(dataset).merge(output['group_labels'], on='example_id')
    annotation_f1 = np.mean([f1_score(merged[f'{gt}_label'], merged[f'group{pred}'], average='macro')
                             for gt, pred in grouping['gt_to_pred_mapping'].items()])

    # Most relevant items: for each pair of groups, are the K top-scored items the K items on which the true groups disagree?
    disagreement = dataset._disagreement_df
    mri_f1 = []
    for column in output['pair_scores'].columns.drop('example_id'):
        a, b = (grouping['pred_to_gt_mapping'][int(g)] for g in column[1:].split('_g'))
        true_ids = disagreement.loc[disagreement['disagreeing_group_pairs'].apply(lambda pairs: (a, b) in pairs or (b, a) in pairs), 'id']
        if len(true_ids) == 0:
            continue
        ranking = output['pair_scores'].sort_values(column, ascending=False)
        y_pred = np.zeros(len(ranking), dtype=int)
        y_pred[:len(true_ids)] = 1
        mri_f1.append(f1_score(ranking['example_id'].isin(true_ids).astype(int), y_pred, average='macro'))

    return {'grouping_accuracy': float(grouping['accuracy']), 'purity': float(grouping['purity']), 'completeness': float(grouping['completeness']),
            'annotation_f1': float(annotation_f1), 'mri_f1': float(np.mean(mri_f1)) if mri_f1 else np.nan}

As an example, a simple method that ignores the texts:
- represent each annotator by how much their labels deviate from each item's average label (0 for items they did not annotate), and cluster these vectors with k-means;
- a group's label for an item is the majority vote of its members (falling back to all annotators if no member annotated the item);
- two groups disagree on an item as much as their shares of "hateful" labels differ.

Replace it with your own method: only the returned dictionary matters.

In [ ]:
def simple_method(train_items: dict, n_groups: int, seed: int = 0) -> dict:
    long = pd.DataFrame([(item_id, annotator, int(label)) for item_id, item in train_items.items()
                         for annotator, label in item['annotations'].items()], columns=['example_id', 'annotator', 'label'])
    long['deviation'] = long['label'] - long.groupby('example_id')['label'].transform('mean')
    matrix = long.pivot_table(index='annotator', columns='example_id', values='deviation', fill_value=0.0)
    clusters = KMeans(n_clusters=n_groups, n_init=10, random_state=seed).fit_predict(matrix.to_numpy())
    annotator_group = dict(zip(matrix.index, clusters))

    long['group'] = long['annotator'].map(annotator_group)
    share = long.pivot_table(index='example_id', columns='group', values='label', aggfunc='mean').reindex(columns=range(n_groups))
    overall = long.groupby('example_id')['label'].mean()
    share = share.apply(lambda column: column.fillna(overall))

    return {
        'groups': {g: [a for a, c in annotator_group.items() if c == g] for g in range(n_groups)},
        'group_labels': (share >= 0.5).astype(int).rename(columns=lambda g: f'group{g}').reset_index(),
        'pair_scores': pd.DataFrame({f'g{a}_g{b}': (share[a] - share[b]).abs()
                                     for a in range(n_groups) for b in range(a + 1, n_groups)}).reset_index(),
    }

n_groups = len(experiments['balanced'][VERSION]['groups'])
evaluate(dataset, simple_method(train_items, n_groups))

{'grouping_accuracy': 1.0,
 'purity': 1.0,
 'completeness': 1.0,
 'annotation_f1': 0.9624676772094136,
 'mri_f1': 0.9048620814873959}

## 4. Running the benchmark

A full evaluation repeats this over experiments, annotator configurations and variants. Here we use the first three experiments of each variant (2 to 4 groups) and two annotator configurations.

In [ ]:
CONFIGURATIONS = [(10, 2), (50, 5)]  # (num_annotators, annotations_per_entry)

results = []
for variant in VALIDATION_VARIANTS:
    for version in list(experiments[variant])[:3]:
        n_groups = len(experiments[variant][version]['groups'])
        for num_annotators, annotations_per_entry in CONFIGURATIONS:
            ds = make_dataset(variant, version, num_annotators, annotations_per_entry)
            results.append({'variant': variant, 'version': version, 'gt_groups': n_groups,
                            'num_annotators': num_annotators, 'annotations_per_entry': annotations_per_entry,
                            **evaluate(ds, simple_method(ds.get_split('train'), n_groups))})
results = pd.DataFrame(results)
results.groupby('variant')[['grouping_accuracy', 'annotation_f1', 'mri_f1']].mean().round(3)

,grouping_accuracy,annotation_f1,mri_f1
variant,,,
balanced,0.800,0.875,0.735
bias,0.683,0.852,0.683
imbalanced,0.850,0.866,0.653
